# 12. Joint Optimization: FWHM + Sigma Matching

**Goal:** jointly optimize μ and γ by matching **both** the FWHM distribution and the fit-uncertainty distribution.

| Gradient | Source | Why |
|---|---|---|
| **μ** (REINFORCE) | Per-quantile reward: −W₁(FWHM) − λ·W₁(σ) | σ encodes photon count statistics directly |
| **γ** (implicit diff) | dLoss/dγ = W₁'(FWHM)·dFWHM/dγ + λ·W₁'(σ)·dσ/dγ | Matching both tightens γ constraints |

**dσ/dγ** is approximated via CRLB: σ ≈ 2γ/√n → dσ/dγ ≈ 2/√n.

All model code from `src/`.

In [ ]:
import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

from src.fitting import (
    _raw_from_width as _rw,
    log_pdf, nll, fwhm_from_theta, fit_profile
)
from src.samplers import draw_fixed_noise, build_photons
from src.implicit import compute_fwhm_and_dgamma

print('Imports OK')

In [ ]:
GAMMA_TRUE = None  # not used for real data
NBAR_TRUE = None  # not used for real data
LAMBDA_ = 2.0

N_TARGET = None  # set from data below
N_RUNS = 200
N_ITER = 80

SIGMA_PROP = 6.0          # physical noise std
LR_MU = 15.0              # learning rate for mu
LR_GAMMA = 0.5            # learning rate for gamma
BASELINE_ALPHA = 0.05     # EMA smoothing
CLIP = 10.0               # gradient clipping

LAMBDA_SIGMA = 0.3        # weight for sigma term in loss/reward
LAMBDA_GAMMA_SIGMA = 0.3  # weight for sigma term in gamma gradient

MU_INIT = 30.0
GAMMA_INIT = 15.0
SEED = 42

print('Parameters set')

## Generate Target Data

Collect both FWHM and sigma_FWHM at true (μ=50, γ=20).

In [ ]:
# ---- Load real experimental data ----
SCALE = 1000.0  # scale factor to match simulator units
df = pd.read_csv('data/processed/fwhm_linewidths.csv')

# Use 3 nW, 40% transmission (most data points)
cond = (df['power_nW'] == 3) & (df['transmission'] == 40)
target_raw = df[cond]['fwhm'].dropna().values * SCALE
target_data = torch.tensor(target_raw, dtype=torch.float32)
N_TARGET = len(target_data)

target_fwhms = target_data.tolist()
target_sigmas = [target_data.std().item()] * N_TARGET  # approximate

print(f'Target: {N_TARGET} FWHM values from real data (3nW, 40% transmission)')
print(f'  Mean: {target_data.mean():.2f}, Std: {target_data.std():.2f}')
print(f'  Range: [{target_data.min():.2f}, {target_data.max():.2f}]')


## Joint Optimization: FWHM + Sigma

### μ update (REINFORCE)
Per-quantile reward combines FWHM and sigma matches, using `adv = loss - baseline` for consistent signs:
```
cl_i = |FWHM_(i) - target_(i)| + λ_sig · |σ_(i) - σ_target_(i)|
adv_i = cl_i - baseline
∇_μ = mean(adv_i × (n_i − μ) / σ²)
```

### γ update (implicit diff + CRLB)
```
∇γ_FWHM  = mean(sign(FWHM_i − target_i) × dFWHM_i/dγ)
∇γ_sigma = mean(sign(σ_i − σ_target_i) × dσ_i/dγ)    # dσ/dγ ≈ 2/√n
∇γ = ∇γ_FWHM + λ · ∇γ_sigma
```

In [ ]:
mu_val = float(MU_INIT)
gamma_val = float(GAMMA_INIT)
bl = 0.0
history = []

print(f"μ_init={MU_INIT}, γ_init={GAMMA_INIT}")
print(f"N_ITER={N_ITER}, N_RUNS={N_RUNS}, λ_sig={LAMBDA_SIGMA}, λ_γ_sig={LAMBDA_GAMMA_SIGMA}\n")

for step in range(N_ITER):
    rng2 = np.random.default_rng(SEED + step)
    fwhms, sigmas, dfs, ns = [], [], [], []
    
    for _ in range(N_RUNS):
        u, b, n = draw_fixed_noise(mu_val, SIGMA_PROP, LAMBDA_, rng2)
        ns.append(n)
        fw, sig, dg = compute_fwhm_and_dgamma(
            gamma_val, u.numpy(), b.numpy(),
            _fit_fn, _fwhm_fn, _nll_fn, n_params=2
        )
        fwhms.append(fw)
        sigmas.append(sig)
        dfs.append(dg)
    
    ft = torch.tensor(fwhms, dtype=torch.float32)
    si_t = torch.tensor(sigmas, dtype=torch.float32)
    nt = torch.tensor(ns, dtype=torch.float32)
    dg_t = torch.tensor(dfs, dtype=torch.float32)
    
    # Sorted quantile matching
    sf, sidx = torch.sort(ft)
    pl = torch.abs(sf - st[:N_RUNS])           # FWHM loss per quantile
    nss = nt[sidx]                               # n sorted by FWHM rank
    dgs = dg_t[sidx]                             # dFWHM/dγ sorted by FWHM rank
    ss = si_t[sidx]                              # σ sorted by FWHM rank
    
    # Sigma loss per quantile (same sort order)
    pl_sig = torch.abs(ss - st_sig[:N_RUNS])
    
    # Combined per-quantile loss (positive, used for both baseline and adv)
    cl = pl + LAMBDA_SIGMA * pl_sig
    
    loss_fwhm = pl.mean()
    loss_sigma = pl_sig.mean()
    mean_loss = (loss_fwhm + LAMBDA_SIGMA * loss_sigma).item()
    loss_fwhm_val = loss_fwhm.item()
    loss_sigma_val = loss_sigma.item()
    
    # Baseline
    if step == 0:
        bl = mean_loss
    else:
        bl = (1 - BASELINE_ALPHA) * bl + BASELINE_ALPHA * mean_loss
    
    # ---- MU gradient: REINFORCE ----
    # FIXED: adv = combined_loss - baseline (consistent signs)
    adv = (cl.detach() - bl).numpy()
    scores = (nss.numpy() - mu_val) / SIGMA_PROP**2
    raw_grad_mu = float(np.mean(adv * scores))
    grad_mu = max(min(raw_grad_mu, CLIP), -CLIP)
    mu_val += LR_MU * (-grad_mu)
    mu_val = max(1.0, min(200.0, mu_val))
    
    # ---- GAMMA gradient: implicit diff (FWHM) + CRLB (sigma) ----
    # dσ/dγ per run: CRLB ≈ 2/√n
    dsigs = torch.tensor([min(2.0 / math.sqrt(max(int(n), 1)), 2.0) for n in ns], dtype=torch.float32)
    ds_s = dsigs[sidx]
    
    signs_fw = torch.sign(sf - st[:N_RUNS])
    signs_sg = torch.sign(ss - st_sig[:N_RUNS])
    raw_grad_fwhm = float((signs_fw * dgs).mean().item())
    raw_grad_sigma = float((signs_sg * ds_s).mean().item())
    raw_grad_gamma = raw_grad_fwhm + LAMBDA_GAMMA_SIGMA * raw_grad_sigma
    grad_gamma = max(min(raw_grad_gamma, CLIP), -CLIP)
    gamma_val += LR_GAMMA * (-grad_gamma)
    gamma_val = max(0.1, min(100.0, gamma_val))
    
    # Diagnostics
    rho = float(np.corrcoef(nss.numpy(), pl.numpy())[0, 1]) if pl.std() > 0.01 and nss.std() > 0.01 else 0.0
    
    info = {
        'step': step,
        'mu': mu_val, 'gamma': gamma_val,
        'loss': mean_loss, 'loss_fwhm': loss_fwhm_val, 'loss_sigma': loss_sigma_val,
        'baseline': bl,
        'grad_mu': grad_mu, 'grad_gamma': grad_gamma,
        'rho': rho, 'mean_n': float(np.mean(ns)),
        'mean_fwhm': float(ft.mean().item()),
        'mean_sigma': float(ss.mean().item()),
    }
    history.append(info)
    
    if step % 5 == 0 or step == N_ITER - 1:
        print(f"  S{step:2d}: μ={mu_val:6.2f} γ={gamma_val:5.1f} | "
              f"L={mean_loss:.2f}(F={loss_fwhm_val:.2f}+S={loss_sigma_val:.2f}) | "
              f"∇μ={grad_mu:+.4f} ∇γ={grad_gamma:+.4f} | "
              f"n̄={info['mean_n']:4.1f} ρ={rho:+.3f} "
              f"({time.time()-t_total:.0f}s)", flush=True)

print(f"\nDone. {time.time()-t_total:.0f}s")

## Results

In [ ]:
if len(history) > 0:
    fmu = history[-1]['mu']
    fga = history[-1]['gamma']
    print(f"{'='*65}")
    print(f"  JOINT OPTIMIZATION — FWHM + SIGMA MATCHING")
    print(f"{'='*65}")
    print(f"  μ:     {MU_INIT:.0f} → {fmu:.2f}  (# no true value\n    print(f"  γ:     {GAMMA_INIT:.0f} → {fga:.2f}  (# no true value\n    print(f"  Loss:  {history[0]['loss']:.2f} → {history[-1]['loss']:.2f}")
    print(f"  FWHM:  {history[0]['loss_fwhm']:.2f} → {history[-1]['loss_fwhm']:.2f}")
    print(f"  Sigma: {history[0]['loss_sigma']:.2f} → {history[-1]['loss_sigma']:.2f}")
    print(f"  μ final grad: {history[-1]['grad_mu']:+.4f}")
    print(f"  Time:  {time.time()-t_total:.0f}s")
    print(f"{'='*65}")

## Visualization for Presentation

### μ and γ convergence + trajectory + FWHM distribution comparison

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from src.losses import wasserstein_loss

steps = [h['step'] for h in history]
mu_hist = [h['mu'] for h in history]
gamma_hist = [h['gamma'] for h in history]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# --- μ convergence ---
ax = axes[0, 0]
ax.plot(steps, mu_hist, 'b-', linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('μ')
ax.set_title('REINFORCE: μ Convergence'); ax.legend(); ax.grid(alpha=0.3)

# --- γ convergence ---
ax = axes[0, 1]
ax.plot(steps, gamma_hist, 'r-', linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('γ')
ax.set_title('Implicit Diff: γ Convergence'); ax.legend(); ax.grid(alpha=0.3)

# --- Loss ---
ax = axes[0, 2]
loss_hist = [h['loss'] for h in history]
bl_hist = [h['baseline'] for h in history]
ax.plot(steps, loss_hist, 'k-', linewidth=2, label='Combined loss')
ax.plot(steps, bl_hist, 'k--', linewidth=1.5, alpha=0.6, label='Baseline')
ax.set_xlabel('Iteration'); ax.set_ylabel('Loss')
ax.set_title('Optimization Loss'); ax.legend(); ax.grid(alpha=0.3)

# --- μ gradient ---
ax = axes[1, 0]
grad_mu_hist = [h['grad_mu'] for h in history]
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.plot(steps, grad_mu_hist, 'b-', linewidth=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('∇μ')
ax.set_title('REINFORCE μ Gradient'); ax.grid(alpha=0.3)

# --- γ gradient ---
ax = axes[1, 1]
grad_gamma_hist = [h['grad_gamma'] for h in history]
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.plot(steps, grad_gamma_hist, 'r-', linewidth=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('∇γ')
ax.set_title('Implicit γ Gradient'); ax.grid(alpha=0.3)

# --- Trajectory (μ vs γ) ---
ax = axes[1, 2]
ax.plot(mu_hist, gamma_hist, 'b.-', linewidth=1.5, markersize=8)
ax.plot(mu_hist[0], gamma_hist[0], 'go', markersize=10, label=f'Start ({MU_INIT},{GAMMA_INIT})')
ax.plot(mu_hist[-1], gamma_hist[-1], 'ro', markersize=10, label=f'End ({mu_hist[-1]:.1f},{gamma_hist[-1]:.1f})')
ax.set_xlabel('μ (mean photon count)'); ax.set_ylabel('γ (HWHM MHz)')
ax.set_title('Optimization Trajectory'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Joint Optimization: REINFORCE + Implicit Diff', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('fig_optimization_path.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final forward pass for FWHM distribution comparison
print('Generating final forward pass for distribution comparison...')
rng_f = np.random.default_rng(999)
final_fwhms = []
for _ in range(500):
    u, b, n = draw_fixed_noise(mu_hist[-1], SIGMA_PROP, LAMBDA_, rng_f)
    photons = build_photons(torch.tensor(gamma_hist[-1], dtype=torch.float32), u, b)
    theta = fit_profile(photons, n_iters=80, model='lorentzian', uniform_bg=False)
    if theta is not None:
        final_fwhms.append(fwhm_from_theta(theta, model='lorentzian').item())
    else:
        final_fwhms.append(2.0 * gamma_hist[-1])
final_t = torch.tensor(final_fwhms, dtype=torch.float32)

# Initial vs Target vs Final KDE
x_grid = np.linspace(0, 100, 500)
kde_target = gaussian_kde(target_t.numpy())
kde_initial = gaussian_kde(init_t.numpy())
kde_final = gaussian_kde(final_t.numpy())
w1_final = wasserstein_loss(final_t, target_t).item()
w1_initial = wasserstein_loss(init_t, target_t).item()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_grid, kde_target(x_grid), 'k-', linewidth=2.5, label=f'Target')
ax.plot(x_grid, kde_initial(x_grid), 'b--', linewidth=2, label=f'Initial (μ={MU_INIT}, γ={GAMMA_INIT}, W1={w1_initial:.0f})')
ax.plot(x_grid, kde_final(x_grid), 'r-', linewidth=2.5, label=f'Final (μ={mu_hist[-1]:.1f}, γ={gamma_hist[-1]:.1f}, W1={w1_final:.1f})')
ax.fill_between(x_grid, kde_final(x_grid), kde_target(x_grid), alpha=0.12, color='gray')
ax.set_xlabel('FWHM (MHz)'); ax.set_ylabel('Density')
ax.set_title('FWHM Distribution: Optimization Result')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_fwhm_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Target FWHM: {target_t.mean():.1f} ± {target_t.std():.1f}')
print(f'Final  FWHM: {final_t.mean():.1f} ± {final_t.std():.1f}')
print(f'Initial FWHM: {init_t.mean():.1f} ± {init_t.std():.1f}')